# 1. Imports

In [1]:
%load_ext autoreload
%autoreload 2

import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 500)
pd.options.plotting.backend = "plotly"

# 2. Data

## 2.1 Données de validation de titres.

In [ ]:

# 4 fichiers csv de validations de titres (1 par trimestre) pour 2025 à combiner

# Chemin du répertoire où se trouvent les fichiers csv à combiner
folder_path_validations = os.path.join("..","data","raw","validations_multiples")

# Récuperer tous les fichiers csv dans ce répertoire
validations = glob.glob(os.path.join(folder_path_validations, "*.csv"))

# List comprehension pour lire et concaténer
validations_df = pd.concat((pd.read_csv(f, sep=";") for f in validations), ignore_index=True)

print(f"Successfully stacked {len(validations)} CSV files into 'validations.csv'!")



Successfully stacked 4 CSV files into 'validations.csv'!


## 2.2 Liste et emplacement des gares / stations.

In [ ]:
# Liste des stations généralisée (une station peut regrouper plusieurs lignes de transport associées)
filepath_stations = os.path.join("..","data","raw","stations","emplacement-des-gares-idf-data-generalisee.csv")
stations_df = pd.read_csv(filepath_stations, sep=";")

# 3. EDA

## 3.1. Données de validation de titres.

### 3.1.1. Vue globale des données.

In [4]:
validations_df.head()

,jour,code_stif_trns,code_stif_res,code_stif_arret,id_zdc,libelle_arret,categorie_titre,nb_vald
0,2025-10-31,800.0,803,454,69662.0,SAULES,Autres titres,1.0
1,2025-10-31,800.0,803,454,69662.0,SAULES,Forfaits courts,188.0
2,2025-10-31,800.0,803,788,60987.0,SAVIGNY SUR ORG,Imagine R,1263.0
3,2025-10-31,800.0,803,793,59842.0,SERMAISE,Amethyste,1.0
4,2025-10-31,800.0,803,793,59842.0,SERMAISE,Forfait Navigo,42.0


Colonnes interessantes pour l'analyse : jour, id_zdc pour faire les jointures, categorie_titre, nb_vald.

In [5]:
print(validations_df.shape)
print(f"Données de validation : Nombre de ligne : {validations_df.shape[0]}, Nombre de colonnes : {validations_df.shape[1]} ")

(1892453, 8)
Données de validation : Nombre de ligne : 1892453, Nombre de colonnes : 8 


In [6]:
validations_df.columns

Index(['jour', 'code_stif_trns', 'code_stif_res', 'code_stif_arret', 'id_zdc',
       'libelle_arret', 'categorie_titre', 'nb_vald'],
      dtype='object')

In [7]:
validations_df.describe()

,code_stif_trns,id_zdc,nb_vald
count,1.892453e+06,1.890831e+06,1.892453e+06
mean,5.020467e+02,9.026463e+04,1.140139e+03
std,3.472100e+02,8.951037e+04,3.084966e+03
min,1.000000e+02,-1.000000e+00,1.000000e+00
25%,1.000000e+02,6.653500e+04,4.400000e+01
50%,8.000000e+02,7.115000e+04,2.150000e+02
75%,8.000000e+02,7.192000e+04,1.016000e+03
max,8.100000e+02,9.999990e+05,9.997000e+04


In [8]:
validations_df.describe(include=object)

,jour,code_stif_res,code_stif_arret,libelle_arret,categorie_titre
count,1892453,1892453,1892453,1892453,1892453
unique,365,18,787,779,8
top,2025-11-04,110,48093,GARE DE LYON,Forfait Navigo
freq,5365,806414,8905,7665,284117


In [9]:
validations_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1892453 entries, 0 to 1892452
Data columns (total 8 columns):
 #   Column           Dtype  
---  ------           -----  
 0   jour             object 
 1   code_stif_trns   float64
 2   code_stif_res    object 
 3   code_stif_arret  object 
 4   id_zdc           float64
 5   libelle_arret    object 
 6   categorie_titre  object 
 7   nb_vald          float64
dtypes: float64(3), object(5)
memory usage: 115.5+ MB


In [10]:
validations_df["categorie_titre"].value_counts()

categorie_titre
Forfait Navigo                  284117
Imagine R                       282766
Forfaits courts                 280786
Amethyste                       270264
NON DEFINI                      248620
Autres titres                   244447
Contrat Solidarité Transport    210239
Contrat Solidarite Transport     71214
Name: count, dtype: int64

Doublon catégorie pour "Solidarité" et "Solidarite"  
NON DEFINI désigne une anomalie chez idfm. Ne pas grouper avec "Autres titres".

### 3.1.2. Data cleaning.

In [ ]:
# Modifier id_zdc de float64 en int64.
validations_df["id_zdc"] = validations_df["id_zdc"].astype(int)

In [ ]:
# Correction "categorie_titre": renommer "Contrat Solidarité Transport" par "Contrat Solidarite Transport"
validations_df["categorie_titre"] = validations_df["categorie_titre"].replace("Contrat Solidarité Transport", "Contrat Solidarite Transport")

In [12]:
validations_df["categorie_titre"].value_counts()

categorie_titre
Forfait Navigo                  284117
Imagine R                       282766
Contrat Solidarite Transport    281453
Forfaits courts                 280786
Amethyste                       270264
NON DEFINI                      248620
Autres titres                   244447
Name: count, dtype: int64

In [13]:
# Valeurs manquantes.
validations_df.isna().sum()

jour                  0
code_stif_trns        0
code_stif_res         0
code_stif_arret       0
id_zdc             1622
libelle_arret         0
categorie_titre       0
nb_vald               0
dtype: int64

1622 valeurs manquantes sur 2 millions pour la colonne id_zdc

In [21]:
# A quoi correspondent les valeurs manquantes ?
validations_df[validations_df["id_zdc"].isnull()]

,jour,code_stif_trns,code_stif_res,code_stif_arret,id_zdc,libelle_arret,categorie_titre,nb_vald
481190,2025-09-22,100.0,ND,ND,NaN,Inconnu,Amethyste,155.0
484136,2025-09-19,100.0,ND,ND,NaN,Inconnu,Amethyste,177.0
484137,2025-09-19,100.0,ND,ND,NaN,Inconnu,Contrat Solidarité Transport,683.0
484138,2025-09-19,100.0,ND,ND,NaN,Inconnu,Imagine R,1388.0
485797,2025-09-24,100.0,ND,ND,NaN,Inconnu,Forfaits courts,537.0
...,...,...,...,...,...,...,...,...
1888840,2025-06-27,100.0,ND,ND,NaN,Inconnu,Forfaits courts,5.0
1888841,2025-06-27,760.0,760,00490909,NaN,Aéroport d'Orly,Contrat Solidarité Transport,841.0
1889842,2025-06-30,760.0,760,00490909,NaN,Aéroport d'Orly,Imagine R,1493.0
1889843,2025-06-30,760.0,760,00490909,NaN,Aéroport d'Orly,NON DEFINI,366.0


In [ ]:
# Attribuer une "id_zdc" à "Aéroport d'Orly" (information depuis la table stations) 
validations_df.loc[validations_df["libelle_arret"] == "Aéroport d'Orly", "id_zdc"] = 63284

In [ ]:
# Supprimer les lignes dont "libelle_arret" == "Inconnu" (séléctionner les lignes où le libellé est différent de "Inconnu" et écraser l'acien dataframe).
validations_df = validations_df[validations_df["libelle_arret"] != "Inconnu"]

In [ ]:
# Pas de doublons.
validations_df.duplicated().sum()

np.int64(0)

## 3.2. Liste et emplacement des gares / stations.

### 3.2.1. Vue globale des données.

In [61]:
stations_df.head()

,geo_point_2d,geo_shape,objectid_1,codeunique,nom_long,nom_so_gar,nom_su_gar,id_ref_zdc,nom_zdc,id_ref_zda,nom_zda,idrefliga,idrefligc,res_com,mode,train,rer,metro,tramway,val,tertrain,terrer,termetro,tertram,terval,exploitant,idf,principal,x,y
0,"48.88218913760796, 2.70601418111842","{""coordinates"": [2.70601418111842, 48.88218913...",5,3,Lagny-Thorigny,NaN,NaN,68494,Lagny - Thorigny,427872,Lagny - Thorigny,A01844,C01730,TRAIN P,TRAIN,1,0,0,0,0,0,0,0,0,0,SNCF,1,0,678439.8038,6.864726e+06
1,"48.83964551726303, 2.655130086641314","{""coordinates"": [2.655130086641314, 48.8396455...",7,7,Torcy,NaN,NaN,68129,Torcy,43207,Torcy,A01856,C01742,RER A,RER,0,1,0,0,0,0,0,0,0,0,RATP,1,0,674687.4504,6.860010e+06
2,"48.79565482051668, 2.6503876698084805","{""coordinates"": [2.650387669808481, 48.7956548...",9,10,Roissy-en-Brie,NaN,NaN,67839,Roissy-en-Brie,46568,Roissy-en-Brie,A01843,C01729,RER E,RER,1,1,0,0,0,0,0,0,0,0,SNCF,1,0,674317.7148,6.855121e+06
3,"48.77077891616031, 2.690252782201096","{""coordinates"": [2.690252782201096, 48.7707789...",10,11,Ozoir-la-Ferrière,NaN,NaN,462934,Ozoir-la-Ferrière,462901,Ozoir-la-Ferrière,A01843,C01729,RER E,RER,1,1,0,0,0,0,0,0,0,0,SNCF,1,0,677235.3132,6.852342e+06
4,"48.960226490735955, 2.9500603827528087","{""coordinates"": [2.950060382752809, 48.9602264...",13,16,Trilport,NaN,NaN,68984,Trilport,47962,Trilport,A01844,C01730,TRAIN P,TRAIN,1,0,0,0,0,0,0,0,0,0,SNCF,1,0,696343.0315,6.873365e+06


Colonnes interessantes pour l'analyse :  
geo_point_2d pour carte  
id_ref_zdc pour les jointures  
nom_zdc  
res_com  
mode  
train, rer, metro, tramway, val pour identifier les modes  
tertrain, terrer, termetro, tertram, terval pour voir l'impact des terminus  
exploitant  
principal 

In [62]:
print(stations_df.shape)
print(f"Données de validation : Nombre de ligne : {stations_df.shape[0]}, Nombre de colonnes : {stations_df.shape[1]} ")

(999, 30)
Données de validation : Nombre de ligne : 999, Nombre de colonnes : 30 


In [63]:
stations_df.columns

Index(['geo_point_2d', 'geo_shape', 'objectid_1', 'codeunique', 'nom_long',
       'nom_so_gar', 'nom_su_gar', 'id_ref_zdc', 'nom_zdc', 'id_ref_zda',
       'nom_zda', 'idrefliga', 'idrefligc', 'res_com', 'mode', 'train', 'rer',
       'metro', 'tramway', 'val', 'tertrain', 'terrer', 'termetro', 'tertram',
       'terval', 'exploitant', 'idf', 'principal', 'x', 'y'],
      dtype='object')

In [64]:
stations_df.describe().T

,count,mean,std,min,25%,50%,75%,max
objectid_1,999.0,5.361041e+02,336.213435,-1.000000,2.265000e+02,5.470000e+02,8.345000e+02,1.101000e+03
codeunique,999.0,3.987828e+04,52563.103408,1.000000,2.585000e+02,5.210000e+02,1.070280e+05,1.171150e+05
id_ref_zdc,999.0,1.108205e+05,117408.947771,59403.000000,6.733850e+04,7.122300e+04,7.246600e+04,4.909190e+05
id_ref_zda,999.0,1.214172e+05,160126.428183,42142.000000,4.328400e+04,4.563000e+04,5.401650e+04,4.940200e+05
train,999.0,2.692693e-01,0.443802,0.000000,0.000000e+00,0.000000e+00,1.000000e+00,1.000000e+00
rer,999.0,2.402402e-01,0.427443,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00
metro,999.0,3.213213e-01,0.467218,0.000000,0.000000e+00,0.000000e+00,1.000000e+00,1.000000e+00
tramway,999.0,2.742743e-01,0.446371,0.000000,0.000000e+00,0.000000e+00,1.000000e+00,1.000000e+00
val,999.0,1.001001e-02,0.099598,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00
tertrain,999.0,3.503504e-02,0.183960,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00


In [65]:
stations_df.describe(include=object).T

,count,unique,top,freq
geo_point_2d,999,999,"48.88218913760796, 2.70601418111842",1
geo_shape,999,999,"{""coordinates"": [2.70601418111842, 48.88218913...",1
nom_long,999,997,Saint-Fargeau,2
nom_so_gar,68,65,Hôtel de Ville,2
nom_su_gar,29,12,VITRY-SUR-SEINE,6
nom_zdc,999,990,NC,3
nom_zda,999,994,Saint-Fargeau,2
idrefliga,999,188,A01842,49
idrefligc,999,190,C01728,49
res_com,999,186,RER D,49


In [77]:
stations_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 999 entries, 0 to 998
Data columns (total 30 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   geo_point_2d  999 non-null    object 
 1   geo_shape     999 non-null    object 
 2   objectid_1    999 non-null    int64  
 3   codeunique    999 non-null    int64  
 4   nom_long      999 non-null    object 
 5   nom_so_gar    68 non-null     object 
 6   nom_su_gar    29 non-null     object 
 7   id_ref_zdc    999 non-null    int64  
 8   nom_zdc       999 non-null    object 
 9   id_ref_zda    999 non-null    int64  
 10  nom_zda       999 non-null    object 
 11  idrefliga     999 non-null    object 
 12  idrefligc     999 non-null    object 
 13  res_com       999 non-null    object 
 14  mode          999 non-null    object 
 15  train         999 non-null    int64  
 16  rer           999 non-null    int64  
 17  metro         999 non-null    int64  
 18  tramway       999 non-null    

### 3.2.2. Data cleaning.

In [78]:
# Valeurs manquantes.
stations_df.isna().sum()

geo_point_2d      0
geo_shape         0
objectid_1        0
codeunique        0
nom_long          0
nom_so_gar      931
nom_su_gar      970
id_ref_zdc        0
nom_zdc           0
id_ref_zda        0
nom_zda           0
idrefliga         0
idrefligc         0
res_com           0
mode              0
train             0
rer               0
metro             0
tramway           0
val               0
tertrain          0
terrer            0
termetro          0
tertram           0
terval            0
exploitant        0
idf               0
principal         0
x                 0
y                 0
dtype: int64

Pas de valeurs manquantes sauf pour nom_so_gar (nom sous gare) et nom_su_gar (nom sur gare) qui est normal : toutes les stations n'ont pas forcémment de sur/sous nom.

In [ ]:
# Remplacer dans "termetro": "METRO 14" par 1 (valeur 1 ou 0 uniquement).
stations_df.loc[stations_df["termetro"] == "METRO 14", "termetro"] = 1

In [80]:
# Modifier termetro de object à int64. 
stations_df["termetro"] = stations_df["termetro"].astype(int)

# 4. Export

Exporter les fichiers en format csv pour SQL Big Query.

In [ ]:
# Sauvegarder les données de validations dans un nouveau csv.
clean_validations_filepath = os.path.join('..','data','processed','validations.csv')
validations_df.to_csv(clean_validations_filepath, index=False)

In [ ]:
# Sauvegarder les données de stations dans un nouveau csv.
clean_stations_filepath = os.path.join('..','data','processed','stations.csv')
stations_df.to_csv(clean_stations_filepath, index=False)